### Rag pipleline - data ingestion to vector DB pipline 

In [ ]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\raghu\AppData\Local\Temp\ipykernel_25368\3728471183.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\raghu\OneDrive\Documents\hr_policies_rag_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### read all the pdf inside the directory
def process_all_pdfs(pdf_directory):
    """processing all the pdf inside the diresctory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    #printing the length of the pdf file 
    print(f"Found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\n processing {pdf_file}")
        # Load the PDF file using PyPDFLoader
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"]=pdf_file.name
                doc.metadata["file_type"]='pdf'
            all_documents.extend(documents)
            print(f"👍 loaded {len(documents)}pages")
        except Exception as e:
            print(f"❌ failed to load {pdf_file}: {e}")
    print(f"\n total documaents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 3 pdf files to process

 processing ..\data\pdf\pdf-sample.pdf
👍 loaded 1pages

 processing ..\data\pdf\sample.pdf
👍 loaded 1pages

 processing ..\data\pdf\student-annotated.pdf
👍 loaded 8pages

 total documaents loaded: 10


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2000-06-29T10:21:08+11:00', 'author': 'cdaily', 'moddate': '2013-10-28T15:24:13-04:00', 'title': 'This is a test PDF file', 'source': '..\\data\\pdf\\pdf-sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'pdf-sample.pdf', 'file_type': 'pdf'}, page_content="Adobe Acrobat PDF Files\nAdobe® Portable Document Format (PDF) is a universal file format that preserves all\nof the fonts, formatting, colours and graphics of any source document, regardless of\nthe application and platform used to create it.\nAdobe PDF is an ideal format for electronic document distribution as it overcomes the\nproblems commonly encountered with electronic file sharing.\n• Anyone, anywhere can open a PDF file. All you need is the free Adobe Acrobat\nReader. Recipients of other file formats sometimes can't open files because they\ndon't have the applications used to create 

In [ ]:
## text splitting into chunks
def split_documents(deocuments, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(deocuments)
    print(f"split {len(deocuments)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\n Example chunk")
        print(f"content: {split_docs[0].page_content}")
        print(f"metadata: {split_docs[0].metadata}")
    return split_docs


In [ ]:
chunks=split_documents(all_pdf_documents)
chunks

split 10 documents into 14 chunks

 Example chunk
content: Adobe Acrobat PDF Files
Adobe® Portable Document Format (PDF) is a universal file format that preserves all
of the fonts, formatting, colours and graphics of any source document, regardless of
the application and platform used to create it.
Adobe PDF is an ideal format for electronic document distribution as it overcomes the
problems commonly encountered with electronic file sharing.
• Anyone, anywhere can open a PDF file. All you need is the free Adobe Acrobat
Reader. Recipients of other file formats sometimes can't open files because they
don't have the applications used to create the documents.
• PDF files always print correctly on any printing device.
• PDF files always display exactly as created, regardless of fonts, software, and
operating systems. Fonts, and graphics are not lost due to platform, software, and
version incompatibilities.
• The free Acrobat Reader is easy to download and can be freely distributed by
anyone

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'Microsoft Word 8.0', 'creationdate': '2000-06-29T10:21:08+11:00', 'author': 'cdaily', 'moddate': '2013-10-28T15:24:13-04:00', 'title': 'This is a test PDF file', 'source': '..\\data\\pdf\\pdf-sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'pdf-sample.pdf', 'file_type': 'pdf'}, page_content="Adobe Acrobat PDF Files\nAdobe® Portable Document Format (PDF) is a universal file format that preserves all\nof the fonts, formatting, colours and graphics of any source document, regardless of\nthe application and platform used to create it.\nAdobe PDF is an ideal format for electronic document distribution as it overcomes the\nproblems commonly encountered with electronic file sharing.\n• Anyone, anywhere can open a PDF file. All you need is the free Adobe Acrobat\nReader. Recipients of other file formats sometimes can't open files because they\ndon't have the applications used to create 

### Embedding the text(chunks) into vector data 
## and vectorDB

In [ ]:
%pip install -q sentence-transformers chromadb
%pip install google-cloud-firestore
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity


Note: you may need to restart the kernel to use updated packages.


c:\Users\raghu\OneDrive\Documents\hr_policies_rag_chatbot\.venv\Scripts\python.exe: No module named pip
c:\Users\raghu\OneDrive\Documents\hr_policies_rag_chatbot\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\raghu\OneDrive\Documents\hr_policies_rag_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import sys
!{sys.executable} -m pip install google-cloud-firestore

c:\Users\raghu\OneDrive\Documents\hr_policies_rag_chatbot\.venv\Scripts\python.exe: No module named pip


In [ ]:
pip show google-cloud-firestore

Note: you may need to restart the kernel to use updated packages.


c:\Users\raghu\OneDrive\Documents\hr_policies_rag_chatbot\.venv\Scripts\python.exe: No module named pip


In [ ]:
class embeddingManager:
    def __init__(self,model_name: str ="all-MiniLM-L6-V2"):
        """
        Initialized the embedding manager 
        Args:
            model_nanme: Huggingface model name from sentence embedding 
            """
        self.model_name=model_name
        self.model= None 
        self.load_model()
    
    def _load_model(self):
        """loading the model for the sentencetransformers"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embedding dimensions:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 

    def generate_embiddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embiddings for the given texts
        Args:
            texts: List of text to generate embiddings
        Returns:
            returns the numpy array with the shapr (len(texts),embedding)
        """
        if not self.model:
            raise ValueError("model not loaded ")

        print(f"Generate embedding for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"generate embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """
        Get the embedding dimension of the model
        Returns:
            returns the embedding dimension of the model
        """
        if not self.model:
            raise ValueError("model not loaded ")
        return self.model.get_sentence_embedding_dimension()

## initialize the embedding manager
embedding_manager=embeddingManager()
embedding_manager

loading embedding model: all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6283.74it/s]


model loaded successfully. Embedding dimensions:384


C:\Users\raghu\AppData\Local\Temp\ipykernel_16372\661173700.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully. Embedding dimensions:{self.model.get_sentence_embedding_dimension()}")


### vector store 

In [2]:
import os
import uuid
from pathlib import Path
from typing import Any, List, Optional

import numpy as np
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure


class VectorStore:
    def __init__(
        self,
        collection_name: str = "pdf_documents",
        project_id: Optional[str] = None,
        database: str = "(default)",
    ):
        """
        Initialize the vector store backed by Firestore.

        Args:
            collection_name: Name of the Firestore collection to store documents in.
            project_id: GCP project ID. If None, uses the default from
                        GOOGLE_APPLICATION_CREDENTIALS / environment.
            database: Firestore database ID. Use "(default)" unless you've
                      created a named database.
        """
        self.collection_name = collection_name
        self.project_id = project_id
        self.database = database
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the Firestore client and collection reference"""
        try:
            self.client = firestore.Client(
                project=self.project_id, database=self.database
            )
            self.collection = self.client.collection(self.collection_name)
            print(f"Vector store initialized. Collection: {self.collection_name}")
            # Firestore has no cheap server-side count() for arbitrary collections
            # without an aggregation query, so we use count() aggregation explicitly.
            count_result = self.collection.count().get()
            existing_count = count_result[0][0].value
            print(f"Existing documents in collection: {existing_count}")
        except Exception as e:
            print(f"ERROR initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store (Firestore).

        Args:
            documents: list of LangChain documents
            embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store...")

        try:
            # Firestore batched writes support up to 500 operations per batch
            batch_size = 500
            total_added = 0

            for batch_start in range(0, len(documents), batch_size):
                batch = self.client.batch()
                batch_docs = list(
                    zip(
                        documents[batch_start: batch_start + batch_size],
                        embeddings[batch_start: batch_start + batch_size],
                    )
                )

                for i, (doc, embedding) in enumerate(batch_docs, start=batch_start):
                    doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"

                    metadata = dict(doc.metadata)
                    metadata["doc_index"] = i
                    metadata["content_length"] = len(doc.page_content)

                    doc_ref = self.collection.document(doc_id)
                    batch.set(
                        doc_ref,
                        {
                            "content": doc.page_content,
                            "embedding": Vector(embedding.tolist()),
                            "metadata": metadata,
                        },
                    )

                batch.commit()
                total_added += len(batch_docs)

            print(f"Successfully added {total_added} documents to vector store")
            count_result = self.collection.count().get()
            print(f"Total documents in collection: {count_result[0][0].value}")

        except Exception as e:
            print(f"ERROR adding documents to vector store: {e}")
            raise

    def similarity_search(
        self, query_embedding: np.ndarray, k: int = 5
    ) -> List[dict]:
        """
        Find the k most similar documents to the given query embedding.

        Args:
            query_embedding: embedding vector for the query
            k: number of results to return

        Returns:
            List of dicts with 'content', 'metadata', and 'vector_distance'
        """
        try:
            results = self.collection.find_nearest(
                vector_field="embedding",
                query_vector=Vector(query_embedding.tolist()),
                distance_measure=DistanceMeasure.COSINE,
                limit=k,
                distance_result_field="vector_distance",
            )

            docs = []
            for doc in results.stream():
                data = doc.to_dict()
                docs.append(
                    {
                        "id": doc.id,
                        "content": data.get("content"),
                        "metadata": data.get("metadata", {}),
                        "vector_distance": data.get("vector_distance"),
                    }
                )
            docs.sort(key=lambda x: x["vector_distance"])
            return docs

        except Exception as e:
            print(f"ERROR querying vector store: {e}")
            raise


# Initialize and test
vectorstore = VectorStore()
vectorstore

ImportError: cannot import name 'firestore' from 'google.cloud' (unknown location)

In [ ]:
#convert the text to embedding
text=[doc.page_content for doc in chunks]

#generate the embeddings 
embeddings = embedding_manager.generate_embiddings(text)

# store in the vectore database
vectorstore.add_documents(chunks,embeddings)

Generate embedding for 14 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.92it/s]

generate embeddings with shape: (14, 384)
Adding 14 documents to the vector store...
Successfully added 14 documents to vector store
Total documents in collection: 14


###Retreaving pipeline from Vectorestore

In [ ]:
class RAGRetreaver:
    """handles query based retrieval form the vectore store"""
    def __init__(self,vector_store: vectorstore, embedding_manager: embeddingManager):
        """ 
        initializing the retriever

        args :
        vectore_store : vectore store containing the documents embeddings
        embedding_manager : manages for generating query embeddings
        """
        self.vectore_store= vectore_store
        self.embedding_manager= embedding_manager
    def retrieve(self, query : str , top_k: int =5, score_threshold: float= 0.0)-> List[Dict[str,Any]]:
        



In [ ]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotevn
load_dotenv()

geoq_api_key=""

llm= ChatGeoq(groq_api_key=)